# VFA-1 — Feature Extractor Comparison

Trains three agents on a 25×25 `FogGridWorld` for **35 000 episodes each** and plots
their learning curves on a single chart.

| Agent | Colour | Parameters |
|-------|--------|------------|
| Tabular Q-Learning | blue | grows every episode (curse of dimensionality) |
| Linear FA — Simple | red | 5 weights |
| Linear FA — Rich | green | 10 weights |

**Scientific question:** does Linear FA generalise across the larger grid while
Tabular Q-Learning degrades due to unseen states?

In [20]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / 'src' / 'environment').exists() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.environment.grid_world import FogGridWorld
from src.agents.tabular_q import TabularQAgent
from src.agents.linear_fa import LinearFAAgent
from src.features.feature_extractor import SimpleFeatureExtractor, RichFeatureExtractor

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Train all three agents

All agents share the same environment object so they encounter identical
random map sequences.  Each episode resets with a new random layout.

Hyperparameters follow the VFA-1 spec:
- `alpha = 0.1` for Tabular (Q-entries are independent)
- `alpha = 0.01` for both FA agents (one theta update shifts every state's estimate)

In [21]:
N_TRAIN = 35_000
GRID    = 25
SMOOTH  = 350

env = FogGridWorld(grid_size=GRID)

In [ ]:
# ── Tabular Q ────────────────────────────────────────────────────────────────
agent_q = TabularQAgent(
    n_actions     = env.action_space.n,
    alpha         = 0.1,
    gamma         = 0.99,
    epsilon_start = 1.0,
    epsilon_end   = 0.05,
    epsilon_decay = 0.99989,  # reaches 0.05 at ~26 250 episodes (75 % of training)
)

q_rewards:     list[float] = []
q_table_sizes: list[int]   = []
q_outcomes:    list[str]   = []

for ep in range(1, N_TRAIN + 1):
    obs, _ = env.reset()
    ep_reward = 0.0
    terminated = truncated = False
    reward = 0.0
    while not (terminated or truncated):
        action = agent_q.select_action(obs)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        agent_q.update(obs, action, reward, next_obs, terminated)
        obs = next_obs
        ep_reward += reward
    agent_q.decay_epsilon()
    q_rewards.append(ep_reward)
    q_table_sizes.append(agent_q.q_table_size)
    if truncated:
        q_outcomes.append('timeout')
    elif reward >= 1.0:
        q_outcomes.append('goal')
    elif reward <= -1.0:
        q_outcomes.append('trap')
    else:
        q_outcomes.append('energy')

    if ep % 5000 == 0:
        recent = q_outcomes[-5000:]
        goal_r   = recent.count('goal')    / 50
        trap_r   = recent.count('trap')    / 50
        energy_r = recent.count('energy')  / 50
        to_r     = recent.count('timeout') / 50
        print(f'[Tabular]   ep {ep:>6}  avg(100) {np.mean(q_rewards[-100:]):+.3f}'
              f'  eps {agent_q.epsilon:.3f}  entries {agent_q.q_table_size:,}'
              f'  | goal {goal_r:.0f}%  trap {trap_r:.0f}%  energy✗ {energy_r:.0f}%  timeout {to_r:.0f}%')

[Tabular]   ep   5000  avg(100) -1.424  eps 0.577  entries 515,186  | goal 5%  trap 16%  energy✗ 79%  timeout 0%


In [ ]:
# ── Linear FA — Simple (5 features) ──────────────────────────────────────────
agent_simple = LinearFAAgent(
    n_actions     = env.action_space.n,
    max_steps     = env.max_steps,
    alpha         = 0.01,
    gamma         = 0.99,
    epsilon_start = 1.0,
    epsilon_end   = 0.05,
    epsilon_decay = 0.99989,  # reaches 0.05 at ~26 250 episodes (75 % of training)
)

simple_rewards:  list[float] = []
simple_outcomes: list[str]   = []

for ep in range(1, N_TRAIN + 1):
    obs, _ = env.reset()
    ep_reward = 0.0
    terminated = truncated = False
    reward = 0.0
    while not (terminated or truncated):
        action = agent_simple.select_action(obs)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        agent_simple.update(obs, action, reward, next_obs, terminated, truncated)
        obs = next_obs
        ep_reward += reward
    agent_simple.decay_epsilon()
    simple_rewards.append(ep_reward)
    if truncated:
        simple_outcomes.append('timeout')
    elif reward >= 1.0:
        simple_outcomes.append('goal')
    elif reward <= -1.0:
        simple_outcomes.append('trap')
    else:
        simple_outcomes.append('energy')

    if ep % 5000 == 0:
        recent = simple_outcomes[-5000:]
        goal_r   = recent.count('goal')    / 50
        trap_r   = recent.count('trap')    / 50
        energy_r = recent.count('energy')  / 50
        to_r     = recent.count('timeout') / 50
        print(f'[Simple FA] ep {ep:>6}  avg(100) {np.mean(simple_rewards[-100:]):+.3f}'
              f'  eps {agent_simple.epsilon:.3f}  theta {np.round(agent_simple.theta, 3)}'
              f'  | goal {goal_r:.0f}%  trap {trap_r:.0f}%  energy✗ {energy_r:.0f}%  timeout {to_r:.0f}%')

In [ ]:
# ── Linear FA — Rich (10 features) ───────────────────────────────────────────
agent_rich = LinearFAAgent(
    n_actions         = env.action_space.n,
    max_steps         = env.max_steps,
    alpha             = 0.01,
    gamma             = 0.99,
    epsilon_start     = 1.0,
    epsilon_end       = 0.05,
    epsilon_decay     = 0.99989,  # reaches 0.05 at ~26 250 episodes (75 % of training)
    feature_extractor = RichFeatureExtractor(),
)

rich_rewards:  list[float] = []
rich_outcomes: list[str]   = []

for ep in range(1, N_TRAIN + 1):
    obs, _ = env.reset()
    ep_reward = 0.0
    terminated = truncated = False
    reward = 0.0
    while not (terminated or truncated):
        action = agent_rich.select_action(obs)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        agent_rich.update(obs, action, reward, next_obs, terminated, truncated)
        obs = next_obs
        ep_reward += reward
    agent_rich.decay_epsilon()
    rich_rewards.append(ep_reward)
    if truncated:
        rich_outcomes.append('timeout')
    elif reward >= 1.0:
        rich_outcomes.append('goal')
    elif reward <= -1.0:
        rich_outcomes.append('trap')
    else:
        rich_outcomes.append('energy')

    if ep % 5000 == 0:
        recent = rich_outcomes[-5000:]
        goal_r   = recent.count('goal')    / 50
        trap_r   = recent.count('trap')    / 50
        energy_r = recent.count('energy')  / 50
        to_r     = recent.count('timeout') / 50
        print(f'[Rich FA]   ep {ep:>6}  avg(100) {np.mean(rich_rewards[-100:]):+.3f}'
              f'  eps {agent_rich.epsilon:.3f}'
              f'  | goal {goal_r:.0f}%  trap {trap_r:.0f}%  energy✗ {energy_r:.0f}%  timeout {to_r:.0f}%')

## 2. Learning-curve comparison

All three smoothed curves on one plot.  The bottom panel shows the
explosive Q-table growth versus the fixed parameter count of both FA agents.

**Interpretation:**
- The Q-table plateaus at a high negative reward because new random maps
  always produce unseen states — learned entries never transfer.
- Both FA curves stabilise at a significantly higher reward because
  `phi(s, a)` generalises across layouts.
- Whether Rich FA improves on Simple FA depends on whether the extra
  features (goal direction, wall density, trap/energy visibility) carry
  signal the 5-feature set misses.

## 1b. Episode outcome rates

How episodes actually end for each agent, smoothed over a rolling window.
The four outcomes are mutually exclusive and sum to 100 % at every point.

In [ ]:
OUTCOME_SMOOTH = 500   # rolling window for outcome rates

def smooth_outcome_rate(outcomes: list[str], label: str, window: int) -> np.ndarray:
    arr = np.array([1.0 if o == label else 0.0 for o in outcomes])
    return np.convolve(arr, np.ones(window) / window, mode='valid') * 100

ep_axis = np.arange(OUTCOME_SMOOTH, N_TRAIN + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, outcomes, title, color in [
    (axes[0], q_outcomes,      'Tabular Q',        'steelblue'),
    (axes[1], simple_outcomes, 'Linear FA — Simple', 'tomato'),
    (axes[2], rich_outcomes,   'Linear FA — Rich',  'seagreen'),
]:
    goal_s   = smooth_outcome_rate(outcomes, 'goal',    OUTCOME_SMOOTH)
    trap_s   = smooth_outcome_rate(outcomes, 'trap',    OUTCOME_SMOOTH)
    energy_s = smooth_outcome_rate(outcomes, 'energy',  OUTCOME_SMOOTH)
    to_s     = smooth_outcome_rate(outcomes, 'timeout', OUTCOME_SMOOTH)

    ax.plot(ep_axis, goal_s,   color='#2ca02c', linewidth=2,   label='Goal reached')
    ax.plot(ep_axis, trap_s,   color='#d62728', linewidth=2,   label='Trap')
    ax.plot(ep_axis, energy_s, color='#ff7f0e', linewidth=2,   label='Energy depleted')
    ax.plot(ep_axis, to_s,     color='#9467bd', linewidth=1.5, linestyle='--', label='Timeout')

    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Episode')
    ax.set_ylim(0, 100)
    ax.set_ylabel('Rate (%)' if ax is axes[0] else '')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'Episode outcome rates  ({OUTCOME_SMOOTH}-ep rolling avg)  —  {GRID}×{GRID} grid',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print(f"{'Agent':<20} {'Goal %':>8} {'Trap %':>8} {'Energy✗ %':>10} {'Timeout %':>10}")
print('-' * 58)
for name, outcomes in [('Tabular Q', q_outcomes), ('Simple FA', simple_outcomes), ('Rich FA', rich_outcomes)]:
    n = len(outcomes)
    print(f"{name:<20} {outcomes.count('goal')/n*100:>7.1f}%"
          f" {outcomes.count('trap')/n*100:>7.1f}%"
          f" {outcomes.count('energy')/n*100:>9.1f}%"
          f" {outcomes.count('timeout')/n*100:>9.1f}%")

In [ ]:
episodes = np.arange(1, N_TRAIN + 1)
kernel   = np.ones(SMOOTH) / SMOOTH
x_smooth = episodes[SMOOTH - 1:]

q_smooth      = np.convolve(q_rewards,      kernel, mode='valid')
simple_smooth = np.convolve(simple_rewards, kernel, mode='valid')
rich_smooth   = np.convolve(rich_rewards,   kernel, mode='valid')

n_simple = agent_simple.feature_extractor.n_features
n_rich   = agent_rich.feature_extractor.n_features

fig, (ax_r, ax_p) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# ── Reward panel ──────────────────────────────────────────────────────────────
ax_r.plot(episodes, q_rewards,      alpha=0.12, color='steelblue', linewidth=0.5)
ax_r.plot(episodes, simple_rewards, alpha=0.12, color='tomato',    linewidth=0.5)
ax_r.plot(episodes, rich_rewards,   alpha=0.12, color='seagreen',  linewidth=0.5)

ax_r.plot(x_smooth, q_smooth,
          color='steelblue', linewidth=2.2,
          label=f'Tabular Q  ({agent_q.q_table_size:,} entries)')
ax_r.plot(x_smooth, simple_smooth,
          color='tomato', linewidth=2.2,
          label=f'Linear FA — Simple  ({n_simple} params)')
ax_r.plot(x_smooth, rich_smooth,
          color='seagreen', linewidth=2.2,
          label=f'Linear FA — Rich  ({n_rich} params)')

ax_r.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax_r.set_ylabel('Episode reward')
ax_r.legend(loc='lower right', fontsize=10)
ax_r.set_title(
    f'Tabular Q vs. Linear FA (Simple & Rich) — '
    f'{GRID}×{GRID} random map, {N_TRAIN} episodes  [{SMOOTH}-ep rolling avg]',
    fontsize=11,
)

# ── Parameter count panel ─────────────────────────────────────────────────────
ax_p.plot(episodes, q_table_sizes,
          color='steelblue', linewidth=1.5, label='Tabular Q-table entries')
ax_p.axhline(n_simple, color='tomato',   linewidth=1.5, linestyle='--',
             label=f'Simple FA ({n_simple} params)')
ax_p.axhline(n_rich,   color='seagreen', linewidth=1.5, linestyle=':',
             label=f'Rich FA ({n_rich} params)')
ax_p.set_ylabel('Parameters')
ax_p.set_xlabel('Episode')
ax_p.legend(fontsize=9)
ax_p.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}')
)

plt.tight_layout()
plt.show()

print(f'Tabular Q     last-100 avg: {np.mean(q_rewards[-100:]):+.4f}  |  {agent_q.q_table_size:,} entries')
print(f'Simple FA     last-100 avg: {np.mean(simple_rewards[-100:]):+.4f}  |  {n_simple} params')
print(f'Rich FA       last-100 avg: {np.mean(rich_rewards[-100:]):+.4f}  |  {n_rich} params')

## 3. Learned weight comparison

Features 0–3 are **action-specific** (differ between UP/DOWN/LEFT/RIGHT).
Features 4+ are state-only (same value for all 4 actions).

| Index | Feature               | Range  | Expected sign |
|-------|-----------------------|--------|---------------|
| 0     | `wall_ahead`          | {0, 1} | − (avoid walls) |
| 1     | `trap_ahead`          | {0, 1} | − (avoid traps) |
| 2     | `moving_toward_goal`  | {0, 1} | + (move toward goal) |
| 3     | `moving_toward_energy`| {0, 1} | + (collect energy) |
| 4     | `low_energy`          | {0, 1} | − (urgency flag, fires at ≤ 20 % energy) |
| 4     | `goal_adjacent`       | {0, 1} | + (goal 1 step away) *(Rich only)* |
| 5     | `energy_adjacent`     | {0, 1} | + (pickup 1 step away) *(Rich only)* |
| 6     | `low_energy`          | {0, 1} | − *(Rich only)* |
| 7     | `walls_norm`          | [0, 1] | context-dependent *(Rich only)* |
| 8     | `free_cells_norm`     | [0, 1] | context-dependent *(Rich only)* |
| 9     | `rho_norm`            | [⅓, 1] | context-dependent *(Rich only)* |

**Simple FA**: 5 params · **Rich FA**: 10 params

In [ ]:
simple_names = [
    'wall_ahead', 'trap_ahead', 'toward_goal',
    'toward_energy', 'low_energy',
]
rich_names = [
    'wall_ahead', 'trap_ahead', 'toward_goal', 'toward_energy',
    'goal_adjacent', 'energy_adjacent',
    'low_energy', 'walls_norm', 'free_cells_norm', 'rho_norm',
]

fig, (ax_s, ax_r) = plt.subplots(1, 2, figsize=(13, 5))

for ax, theta, names, title, color in [
    (ax_s, agent_simple.weights, simple_names, 'Simple FA  (5 params)', 'tomato'),
    (ax_r, agent_rich.weights,   rich_names,   'Rich FA  (10 params)',  'seagreen'),
]:
    colors = [color if w >= 0 else '#555555' for w in theta]
    bars = ax.barh(names, theta, color=colors, edgecolor='#333', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.8)
    for bar, val in zip(bars, theta):
        ax.text(
            val + (0.002 if val >= 0 else -0.002),
            bar.get_y() + bar.get_height() / 2,
            f'{val:+.4f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=8,
        )
    ax.set_xlabel('Weight value')
    ax.set_title(title, fontsize=11)

plt.suptitle('Learned theta after 35 000 episodes', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 4. Episode animation

Runs one episode on the same random map for all three agents simultaneously.

**Layout — 4 columns × 4 rows:**
| Row | Col 0 | Col 1 | Col 2 | Col 3 |
|-----|-------|-------|-------|-------|
| 1 | Legend | Full map — Tabular Q | Full map — Simple FA | Full map — Rich FA |
| 2 | Tabular Q fog view | Path on full map | Energy over time | Stats |
| 3 | Simple FA fog view | Path on full map | Energy over time | Stats |
| 4 | Rich FA fog view | Path on full map | Energy over time | Stats |

Each agent uses its **trained policy at ε = ε_end** (near-greedy; same floor used during training).

In [ ]:
import matplotlib.gridspec as gridspec
from matplotlib.animation import FuncAnimation
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from IPython.display import HTML

# ── Colour map (mirrors FogGridWorld._PALETTE) ───────────────────────────────
_ANIM_PAL = [
    [ 28/255,  28/255,  28/255],   # 0 FOG
    [240/255, 240/255, 224/255],   # 1 FREE
    [ 96/255,  96/255,  96/255],   # 2 WALL
    [204/255,  51/255,  51/255],   # 3 TRAP
    [ 51/255, 204/255,  85/255],   # 4 GOAL
    [255/255, 170/255,   0/255],   # 5 ENERGY
    [ 51/255, 153/255, 255/255],   # 6 AGENT
]
_ANIM_CMAP = ListedColormap(_ANIM_PAL)

ANIM_SEED   = 42
anim_labels = ['Tabular Q', 'Linear FA — Simple', 'Linear FA — Rich']
anim_colors = ['steelblue', 'tomato', 'seagreen']

# ── Three envs with the same seed → identical starting map ───────────────────
anim_envs   = [FogGridWorld(grid_size=GRID) for _ in range(3)]
anim_agents = [agent_q, agent_simple, agent_rich]

obs_list = [env.reset(seed=ANIM_SEED)[0] for env in anim_envs]
ref_grid = anim_envs[0]._grid.copy()   # full revealed map

# Use epsilon_end (the trained floor, e.g. 0.05) rather than 0.
# At epsilon=0 all features zero → identical Q-values → deterministic cycle.
# epsilon_end gives the same small random kick the agent had at convergence.
_saved_eps = [a.epsilon for a in anim_agents]
for a in anim_agents:
    a.epsilon = a.epsilon_end

done_list   = [False] * 3
energy_hist = [[env._energy]    for env in anim_envs]
cumrew_hist = [[0.0]            for _   in range(3)]
path_hist   = [[env._agent_pos] for env in anim_envs]

# ── Figure / GridSpec ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 18))
gs  = gridspec.GridSpec(4, 4, figure=fig, hspace=0.55, wspace=0.35)

ax_leg  = fig.add_subplot(gs[0, 0])
ax_full = [fig.add_subplot(gs[0, i + 1]) for i in range(3)]

fog_ax    = [fig.add_subplot(gs[r, 0]) for r in range(1, 4)]
path_ax   = [fig.add_subplot(gs[r, 1]) for r in range(1, 4)]
energy_ax = [fig.add_subplot(gs[r, 2]) for r in range(1, 4)]
stats_ax  = [fig.add_subplot(gs[r, 3]) for r in range(1, 4)]

# ── Legend ────────────────────────────────────────────────────────────────────
ax_leg.axis('off')
_leg_labels = ['FOG', 'FREE', 'WALL', 'TRAP', 'GOAL', 'ENERGY', 'AGENT']
ax_leg.legend(
    handles=[mpatches.Patch(facecolor=_ANIM_PAL[i], edgecolor='#333',
                             linewidth=0.5, label=_leg_labels[i]) for i in range(7)],
    loc='center', fontsize=9, title='Cell types', title_fontsize=9, framealpha=0.9,
)

# ── Helper ────────────────────────────────────────────────────────────────────
def _draw(ax, grid, agent_pos, rho, title, fog):
    r_a, c_a = agent_pos
    disp = grid.astype(float).copy()
    if fog:
        rows_ = np.arange(GRID)
        cols_ = np.arange(GRID)
        mask  = (np.abs(rows_[:, None] - r_a) > rho) | (np.abs(cols_[None, :] - c_a) > rho)
        disp[mask] = 0.0
    disp[r_a, c_a] = 6.0
    ax.imshow(disp, cmap=_ANIM_CMAP, vmin=0, vmax=6, interpolation='nearest')
    ax.set_title(title, fontsize=8, pad=3)
    ax.set_xticks([]); ax.set_yticks([])

# ── Static row 0: full maps at start ─────────────────────────────────────────
for i, (ax, env) in enumerate(zip(ax_full, anim_envs)):
    _draw(ax, ref_grid, env._agent_pos, env._visible_radius(),
          f'{anim_labels[i]}\nstart position', fog=False)

# ── Animation ─────────────────────────────────────────────────────────────────
_step      = [0]
_eps_reset = [False]

def _update(frame):
    _step[0] += 1

    for i in range(3):
        if done_list[i]:
            continue

        env = anim_envs[i]
        obs_list[i], reward, term, trunc, _ = env.step(
            anim_agents[i].select_action(obs_list[i])
        )
        done_list[i] = term or trunc

        energy_hist[i].append(env._energy)
        cumrew_hist[i].append(cumrew_hist[i][-1] + reward)
        path_hist[i].append(env._agent_pos)
        rho = env._visible_radius()

        # Col 0 — fog view
        fog_ax[i].clear()
        _draw(fog_ax[i], env._grid.copy(), env._agent_pos, rho,
              f'{anim_labels[i]}  |  fog view  (step {_step[0]})', fog=True)

        # Col 1 — full map + path
        path_ax[i].clear()
        _draw(path_ax[i], ref_grid, env._agent_pos, rho,
              f'path  (step {_step[0]})', fog=False)
        if len(path_hist[i]) > 1:
            path_ax[i].plot(
                [p[1] for p in path_hist[i]],
                [p[0] for p in path_hist[i]],
                color=anim_colors[i], lw=1.2, alpha=0.75,
            )

        # Col 2 — energy over time
        energy_ax[i].clear()
        energy_ax[i].plot(energy_hist[i], color=anim_colors[i], lw=1.5)
        energy_ax[i].axhline(env.E_max * 0.3, color='orange', lw=0.8,
                              ls='--', label='30 % threshold')
        energy_ax[i].set_xlim(0, env.max_steps)
        energy_ax[i].set_ylim(0, env.E_max)
        energy_ax[i].set_xlabel('Step', fontsize=7)
        energy_ax[i].set_ylabel('Energy', fontsize=7)
        energy_ax[i].tick_params(labelsize=6)
        energy_ax[i].set_title('Energy level', fontsize=8)

        # Col 3 — text stats
        stats_ax[i].clear()
        stats_ax[i].axis('off')
        if done_list[i]:
            if   reward >= 1.0:  status = 'GOAL  ✓'
            elif reward <= -1.0: status = 'DEAD  ✗'
            else:                status = 'TIMEOUT'
        else:
            status = 'running…'
        stats_ax[i].text(
            0.08, 0.5,
            f"Agent:   {anim_labels[i]}\n"
            f"Step:    {_step[0]}\n"
            f"Energy:  {env._energy} / {env.E_max}\n"
            f"Reward:  {cumrew_hist[i][-1]:+.3f}\n"
            f"Status:  {status}",
            transform=stats_ax[i].transAxes,
            fontsize=9, va='center', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='#f0f0f0', alpha=0.85),
        )

    # Stop + restore ε when all agents are done
    if all(done_list) and not _eps_reset[0]:
        _eps_reset[0] = True
        anim_obj.event_source.stop()
        for a, eps in zip(anim_agents, _saved_eps):
            a.epsilon = eps

    return []

anim_obj = FuncAnimation(
    fig, _update,
    frames=anim_envs[0].max_steps,
    interval=120,
    blit=False,
)
_eps_end_val = anim_agents[1].epsilon_end
fig.suptitle(
    f'Episode animation — {GRID}×{GRID} grid  (seed {ANIM_SEED})  |  ε = {_eps_end_val} (trained floor)',
    fontsize=12, y=1.005,
)
plt.close(fig)
HTML(anim_obj.to_jshtml())